
# Tutorial 2: Symbolic Regression

### Goal:
- Learn an explicit equation relating inputs → outputs
- Compare symbolic regression to a black-box model
- Understand accuracy vs interpretability


In [ ]:
from skmatter.datasets import load_who_dataset
import matplotlib.pyplot as plt
import numpy as np
from utils import parity_plot

data = load_who_dataset()['data']

numeric_data = data.select_dtypes(include=[np.number])
X = numeric_data.drop(columns=["NY.GDP.PCAP.CD"]).values
y = data["NY.GDP.PCAP.CD"].values

print("X shape:", X.shape)
print("y (first 5):", y[:5])

## Symbolic Regression

Symbolic regression searches over mathematical expressions to find:

$$ f(x) \approx y$$

Unlike standard ML:
- It does **not assume a fixed model form**
- It outputs an **explicit equation**

In [ ]:
from pysr import PySRRegressor

model = PySRRegressor(
    niterations=20,
    binary_operators=["+", "*", "-"],
    unary_operators=["sin", "cos"],
    verbosity=0
)

model.fit(X, y)

print("Symbolic regression complete")

In [ ]:
# ==========================================
# TODO: try removing or adding an operator
# ==========================================

In [ ]:
print("Learned equation:")
print(model)

feature_names = numeric_data.drop(columns=["NY.GDP.PCAP.CD"]).columns

print("Feature mapping:")
for i, name in enumerate(feature_names):
    print(f"x{i}: {name}")

In [ ]:
df = model.equations_

# Extract relevant columns
complexity = df["complexity"]
loss = df["loss"]

r2 = 1 - loss / np.var(y)

plt.figure(figsize=(6,4))

plt.scatter(complexity, r2, alpha=0.7, edgecolor="k")

# Highlight best model
best_idx = df["loss"].idxmin()
plt.scatter(
    complexity[best_idx],
    r2[best_idx],
    color="red",
    s=100,
    label="Best model"
)

plt.xlabel("Model Complexity")
plt.ylabel("$R^2$")
plt.title("Pareto Front: Accuracy vs Complexity")

plt.grid(alpha=0.3)
plt.legend()

plt.show()

In [ ]:
# Get best equation (lowest loss)
best_idx = model.equations_["loss"].idxmin()

best_eq = model.equations_.loc[best_idx]

print("Best equation:")
print(best_eq["equation"])

y_sr = model.predict(X)

parity_plot(y, y_sr, title="Symbolic Regression (Best Model)")


## Compare with a Black-Box Model

We now compare symbolic regression to a Random Forest.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor()
rf.fit(X, y)

print("Random Forest trained")
parity_plot(y, rf.predict(X), title="Symbolic Regression (Best Model)")

In [ ]:
rf_score = rf.score(X, y)
sr_score = model.score(X, y)

print("Random Forest R²:", rf_score)
print("Symbolic Regression R²:", sr_score)

In [ ]:
# ==========================================
# TODO: improve symbolic regression
# ==========================================

# Try increasing iterations

# model = PySRRegressor(
#     niterations=50,
#     binary_operators=["+", "*", "-"],
#     unary_operators=["sin", "cos"],
#     verbosity=0
# )
# model.fit(X, y)
# print(model)

## Reflection

Answer these before moving on:

- Which model is more accurate?
- Which model is easier to interpret?
- Does the symbolic model capture key trends?
- Would you trust this equation scientifically?


### Takeaway

- Symbolic regression gives **explicit equations**
- SHAP explains **feature contributions**
- Saliency gives **local sensitivity**

Each method provides a different type of insight.